## 1. Carga y limpieza de datos

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("insurance.csv")

print("Dimensiones iniciales:", df.shape)
display(df.head())
print("Tipos de datos:")
print(df.dtypes)
print("Valores nulos por columna:")
print(df.isnull().sum())

Dimensiones iniciales: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Tipos de datos:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object
Valores nulos por columna:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [3]:
for columna in df.columns:
    if df[columna].isnull().sum() > 0:
        if df[columna].dtype == "object":
            df[columna] = df[columna].fillna(df[columna].mode()[0])
        else:
            df[columna] = df[columna].fillna(df[columna].median())

columnas_numericas = ["age", "bmi", "children", "charges"]
for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

df = df.dropna()
df = df.drop_duplicates()

for columna in ["age", "bmi", "charges"]:
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    df = df[(df[columna] >= limite_inferior) & (df[columna] <= limite_superior)]

df = df.reset_index(drop=True)
print("Dimensiones después de limpiar:", df.shape)
display(df.head())

Dimensiones después de limpiar: (1190, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Análisis exploratorio y agrupaciones

In [4]:
resumen = df.groupby(["region", "smoker"]).agg(
    coste_medio=("charges", "mean"),
    coste_mediano=("charges", "median"),
    desviacion_estandar=("charges", "std"),
    cantidad=("charges", "count")
).reset_index()

display(resumen.round(2))

,region,smoker,coste_medio,coste_mediano,desviacion_estandar,cantidad
0,northeast,no,9164.49,8338.75,6169.23,256
1,northeast,yes,21086.98,20296.86,5753.06,39
2,northwest,no,8582.47,7259.23,6125.33,266
3,northwest,yes,23061.06,21725.80,5607.32,38
4,southeast,no,7952.81,6593.51,5924.35,267
5,southeast,yes,22019.11,19629.47,4682.45,34
6,southwest,no,7807.84,7345.08,4936.21,265
7,southwest,yes,20867.65,19199.94,5032.21,25


In [5]:
df["coste_medio_grupo"] = df.groupby(["region", "smoker"])["charges"].transform("mean")
df["desviacion_grupo"] = df.groupby(["region", "smoker"])["charges"].transform("std")

df["diferencia_coste"] = df["charges"] - df["coste_medio_grupo"]

df["porcentaje_ahorro"] = (
    (df["coste_medio_grupo"] - df["charges"]) / df["coste_medio_grupo"] * 100
)

display(df[["region", "smoker", "charges", "coste_medio_grupo",
            "diferencia_coste", "porcentaje_ahorro"]].head())

,region,smoker,charges,coste_medio_grupo,diferencia_coste,porcentaje_ahorro
0,southwest,yes,16884.92400,20867.650121,-3982.726121,19.085647
1,southeast,no,1725.55230,7952.805605,-6227.253305,78.302597
2,southeast,no,4449.46200,7952.805605,-3503.343605,44.051669
3,northwest,no,21984.47061,8582.467101,13402.003509,-156.155606
4,northwest,no,3866.85520,8582.467101,-4715.611901,54.944713


## 3. Motor de IA basado en reglas estadísticas

In [6]:
def clasificar_fila(fila):
    media = fila["coste_medio_grupo"]
    desviacion = fila["desviacion_grupo"]
    coste = fila["charges"]

    if pd.isna(desviacion):
        return "Precio de Mercado"
    elif coste < media - desviacion:
        return "Oportunidad"
    elif coste > media + desviacion:
        return "Sobrevalorado"
    else:
        return "Precio de Mercado"

df["clasificacion"] = df.apply(clasificar_fila, axis=1)

print("Número de registros por clasificación:")
print(df["clasificacion"].value_counts())
display(df[["age", "bmi", "region", "smoker", "charges",
            "coste_medio_grupo", "clasificacion"]].head(10))

Número de registros por clasificación:
clasificacion
Precio de Mercado    885
Oportunidad          165
Sobrevalorado        140
Name: count, dtype: int64


,age,bmi,region,smoker,charges,coste_medio_grupo,clasificacion
0,19,27.900,southwest,yes,16884.92400,20867.650121,Precio de Mercado
1,18,33.770,southeast,no,1725.55230,7952.805605,Oportunidad
2,28,33.000,southeast,no,4449.46200,7952.805605,Precio de Mercado
3,33,22.705,northwest,no,21984.47061,8582.467101,Sobrevalorado
4,32,28.880,northwest,no,3866.85520,8582.467101,Precio de Mercado
5,31,25.740,southeast,no,3756.62160,7952.805605,Precio de Mercado
6,46,33.440,southeast,no,8240.58960,7952.805605,Precio de Mercado
7,37,27.740,northwest,no,7281.50560,8582.467101,Precio de Mercado
8,37,29.830,northeast,no,6406.41070,9164.487165,Precio de Mercado
9,60,25.840,northwest,no,28923.13692,8582.467101,Sobrevalorado


## 4. Generación de reportes automatizados

In [7]:
oportunidades = df[df["clasificacion"] == "Oportunidad"].copy()
oportunidades = oportunidades.sort_values(
    by="porcentaje_ahorro", ascending=False
)

mejores_oportunidades = oportunidades.head(20)

display(mejores_oportunidades[[
    "age", "bmi", "children", "smoker", "region", "charges",
    "coste_medio_grupo", "porcentaje_ahorro", "clasificacion"
]].round(2))

,age,bmi,children,smoker,region,charges,coste_medio_grupo,porcentaje_ahorro,clasificacion
838,18,23.21,0,no,southeast,1121.87,7952.81,85.89,Oportunidad
720,18,30.14,0,no,southeast,1131.51,7952.81,85.77,Oportunidad
1106,18,33.33,0,no,southeast,1135.94,7952.81,85.72,Oportunidad
591,18,33.66,0,no,southeast,1136.40,7952.81,85.71,Oportunidad
20,18,34.10,0,no,southeast,1137.01,7952.81,85.70,Oportunidad
171,18,34.43,0,no,southeast,1137.47,7952.81,85.70,Oportunidad
768,18,37.29,0,no,southeast,1141.45,7952.81,85.65,Oportunidad
694,18,41.14,0,no,southeast,1146.80,7952.81,85.58,Oportunidad
388,18,43.01,0,no,southeast,1149.40,7952.81,85.55,Oportunidad
1112,19,19.80,0,no,southwest,1241.56,7807.84,84.10,Oportunidad


In [8]:
df.to_csv("informe_limpio_clasificado.csv", index=False)
mejores_oportunidades.to_csv("mejores_oportunidades.csv", index=False)

print("Archivos generados:")
print("- informe_limpio_clasificado.csv")
print("- mejores_oportunidades.csv")

Archivos generados:
- informe_limpio_clasificado.csv
- mejores_oportunidades.csv
